<a href="https://colab.research.google.com/github/VICO-27/colab-notebooks/blob/main/EDA_for_GeoAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA) Report – Progress 1

## Dataset Overview
- Training set: **1,821 samples × 146 columns**
- Test set: **1,030 samples × 145 columns**
- Target variable: `label` (0 = Other land, 1 = Aquaculture pond)
- Features: **144 satellite-derived features** representing **12 spectral/radar bands across 12 months**.

---

## Data Quality Assessment

### Missing Values
- No `NaN` values were found in either the training or test datasets.
- The competition uses **-9999** as a sentinel value to represent missing satellite observations rather than actual missing values.
- These values should be treated as missing during preprocessing instead of real measurements.

### Duplicate Records
- No duplicate samples were found in either dataset.
- The dataset is clean and requires minimal preprocessing.

---

## Target Distribution

| Class | Count | Percentage |
|-------|------:|-----------:|
| Other Land (0) | 1086 | 59.6% |
| Aquaculture Pond (1) | 735 | 40.4% |

### Insight
The dataset is only moderately imbalanced (60:40). This imbalance is unlikely to require aggressive resampling techniques, but using class weighting in machine learning models may improve performance.

---

## Feature Analysis

The dataset consists of two sensor types:

- **Sentinel-1 (Radar):** VH, VV
- **Sentinel-2 (Optical):** Blue, Green, Red, Red Edge (RE1–RE3), NIR, Narrow NIR, SWIR1, SWIR2

These sensors capture complementary information: radar measures surface structure and moisture, while optical bands capture spectral reflectance.

---

## Feature Separability

Analysis of class-wise feature distributions shows:

- **Green band** exhibits the clearest individual separation between ponds and non-ponds.
- **NIR, Narrow NIR, SWIR1, RE2, and RE3** show large differences in class means, confirming that infrared wavelengths are highly informative for distinguishing water from vegetation.
- **VH and VV** provide useful structural information but have considerable overlap between classes.
- **Red band** provides very little discrimination when used alone.

### Insight
No single feature perfectly separates the two classes. The classification problem depends on combining multiple complementary features rather than relying on individual bands.

---

## Correlation Analysis

Several highly correlated feature groups were identified:

- NIR ↔ Narrow NIR ↔ RE2 ↔ RE3
- SWIR1 ↔ SWIR2
- Blue ↔ Green

### Insight
These bands measure similar spectral characteristics and therefore contain redundant information. While this may affect linear models, tree-based algorithms such as LightGBM, CatBoost, and XGBoost can naturally handle correlated features.

---

## Key Findings

- The dataset is clean with no duplicates or true missing values.
- Moderate class imbalance suggests using class-aware learning rather than oversampling.
- Infrared-based bands (NIR, Narrow NIR, SWIR1, RE2, RE3) provide the strongest discrimination between ponds and other land covers.
- Green is the strongest standalone visible band.
- Radar features contribute complementary information despite overlapping distributions.
- Strong correlations exist among several optical bands, reflecting similar physical properties rather than data quality issues.

---

## Preliminary Conclusions

The initial EDA indicates that successful models should:
- Combine radar and optical information.
- Exploit infrared-based bands and derived water indices.
- Use tree-based ensemble models capable of learning feature interactions.
- Properly handle the temporal nature of the dataset during feature engineering.

## Temporal EDA – NIR Seasonal Analysis

### Objective
Investigate how Near Infrared (NIR) reflectance changes throughout the year for aquaculture ponds and other land-cover classes.

### Findings
- The "Other" class exhibits a strong seasonal pattern, with NIR increasing steadily from January and peaking in August before declining toward December.
- Aquaculture ponds maintain consistently lower NIR values throughout the year with noticeably smaller seasonal variation.
- The separation between the two classes becomes much larger during the middle of the year, particularly from April to August.
- The two curves never intersect, indicating a stable distinction between ponds and non-ponds across all months.

### Interpretation
The observed behavior aligns with remote sensing theory. Vegetation strongly reflects Near Infrared radiation during its growing season, while water absorbs NIR, resulting in consistently lower reflectance values for ponds.

### Modeling Implications
- NIR is one of the most informative bands for this classification task.
- Temporal aggregation features such as mean and standard deviation of NIR are expected to improve model robustness.
- Seasonal variability itself is likely to be an informative feature for distinguishing vegetation from permanent water bodies.

## Temporal EDA – Multi-Band Seasonal Analysis

### Objective
Analyze how key radar and optical bands change throughout the year for aquaculture ponds and other land-cover classes.

### Findings

- **Green:** Aquaculture ponds consistently exhibit higher green reflectance than other land-cover types across all months, likely due to water properties such as turbidity, suspended sediments, or algae.
- **NIR:** Other land-cover classes show a strong seasonal increase in NIR reflectance, peaking during the middle of the year, while ponds remain consistently lower because water absorbs Near Infrared radiation.
- **SWIR1:** SWIR1 displays one of the strongest separations between classes. Ponds maintain low SWIR reflectance throughout the year, whereas vegetation shows much higher values, especially during peak growing months.
- **VH Radar:** Unlike optical bands, VH measures surface roughness rather than reflected light. Although seasonal variation exists, it follows a different pattern and provides complementary structural information.

### Key Insight

Each sensor captures different physical characteristics of the Earth's surface. Optical bands primarily distinguish vegetation from water through spectral reflectance, while radar provides information about surface texture and moisture. Their complementary nature suggests that combining both sensor types will improve classification performance.

### Modeling Implications

- Retain both radar and optical features.
- Prioritize infrared bands (NIR and SWIR1) during feature engineering.
- Compute temporal aggregation statistics (mean, standard deviation, minimum, maximum, range) for each band.
- Develop water-related spectral indices (NDWI, MNDWI, NDVI) to further enhance class separability.

## Temporal EDA – Spectral Indices Analysis

### Objective
Evaluate whether remote sensing spectral indices provide better class separation than individual satellite bands.

### Findings

Three standard spectral indices were computed for Month 01:

| Index | Pond Mean | Other Mean | Difference |
|--------|----------:|-----------:|-----------:|
| NDVI | 0.038 | 0.117 | -0.080 |
| NDWI | -0.010 | -0.134 | +0.124 |
| MNDWI | 0.005 | -0.147 | +0.152 |

### Interpretation

- **NDVI** successfully distinguishes vegetation from water but provides the weakest separation among the three indices.
- **NDWI** improves class separation by combining the Green and NIR bands, effectively highlighting water bodies.
- **MNDWI** achieves the largest separation, confirming that replacing NIR with SWIR1 enhances water detection due to the strong absorption of SWIR by water.

### Key Insight

The spectral indices outperform individual raw bands by combining complementary spectral information into physically meaningful features. Among the evaluated indices, **MNDWI** appears to be the strongest candidate for feature engineering.

### Modeling Implications

- Generate NDVI, NDWI, and MNDWI for all twelve months.
- Aggregate each index across valid months using statistics such as mean, standard deviation, minimum, and maximum.
- Include all three indices in the final feature set, allowing tree-based models to determine their relative importance.

## Temporal Feature Engineering – NIR Aggregation

### Objective

Evaluate whether temporal aggregation statistics provide a more robust representation of yearly NIR behavior than individual monthly observations.

### Results

| Feature | Pond | Other | Difference |
|---------|------:|------:|-----------:|
| NIR Mean | 1893.73 | 2939.90 | -1046.18 |
| NIR Std | 559.02 | 729.08 | -170.07 |
| NIR Min | 1243.43 | 2041.46 | -798.03 |
| NIR Max | 2933.72 | 4243.19 | -1309.47 |
| NIR Range | 1690.29 | 2201.73 | -511.44 |

### Findings

- **NIR Mean** effectively summarizes the annual spectral behavior and provides strong class separation.
- **NIR Standard Deviation** confirms that vegetation exhibits greater seasonal variability than aquaculture ponds.
- **NIR Minimum** captures the consistently low infrared reflectance of water.
- **NIR Maximum** produces the largest class separation, reflecting the strong seasonal growth of vegetation.
- **NIR Range** highlights that vegetation undergoes larger annual changes than ponds.

### Key Insight

Temporal aggregation transforms twelve monthly observations into a compact and informative representation while preserving seasonal characteristics. These aggregated features are also robust to the partial observation windows present in the test set.

### Modeling Implications

- Generate temporal aggregation statistics for all important bands.
- Prioritize aggregated features over individual monthly values.
- Apply the same strategy to spectral indices (NDVI, NDWI, and MNDWI) to improve robustness against the train–test temporal mismatch.

## Correlation Analysis

### Objective
Identify redundant features and understand relationships between satellite bands.

### Findings

- Four strong correlation groups were identified:
  - **Radar:** VH and VV (r = 0.95)
  - **Visible:** Blue, Green, Red (r = 0.89–0.96)
  - **Vegetation/Red-edge:** RE2, RE3, NIR, NIRA (r = 0.99–1.00)
  - **SWIR:** SWIR1 and SWIR2 (r = 0.98)

- Radar bands exhibit very low correlation with the visible spectrum, indicating that SAR and optical imagery provide complementary information.

- RE1 acts as a transition band between visible and infrared wavelengths.

### Implications

- High correlation does not necessarily mean features should be removed because tree-based models (CatBoost, LightGBM, XGBoost) handle correlated features well.
- Instead of feature removal, correlation information will guide feature engineering and later feature importance analysis.

# Exploratory Data Analysis (EDA) Report – Progress 2

---

## Feature Importance Analysis

### Objective
Identify which features contribute the most to distinguishing aquaculture ponds from other land cover using a baseline Random Forest model.

### Method
- Trained a Random Forest classifier on the original monthly features.
- Computed feature importance scores for every feature.
- Ranked features according to their contribution to classification.

### Findings

The most influential features were:

| Rank | Feature |
|------|---------|
| 1 | nir_07 |
| 2 | re2_08 |
| 3 | re3_08 |
| 4 | nir_08 |
| 5 | nira_07 |
| 6 | re2_07 |
| 7 | swir1_06 |
| 8 | nira_08 |
| 9 | swir1_08 |
|10 | swir1_07 |

### Key Observations

- Most important features come from **July (07)** and **August (08)**.
- Infrared (NIR, NIRA), Red-Edge (RE2, RE3), and SWIR bands dominate the ranking.
- Visible bands contribute less.
- Radar features appear less frequently among the top-ranked features.

### Interpretation

The model relies heavily on seasonal vegetation and water characteristics during mid-year. However, since the test dataset contains varying seasonal windows (4–6 months only), relying solely on raw monthly features may lead to overfitting.

### Decision

Do **not** remove raw monthly features.

Instead, combine:

- Raw monthly features
- Spectral indices
- Temporal aggregated features

to improve robustness against missing seasonal observations.

---

# Feature Engineering Pipeline

## Objective

Build a reusable feature engineering pipeline that automatically transforms raw satellite observations into machine-learning-ready features for both training and testing datasets.

The pipeline is modular so that identical preprocessing is applied to Train and Test data.

---

## Step 1 — Band Definitions

Defined the satellite bands used throughout the project.

### Radar Bands

- VH
- VV

### Optical Bands

- Blue
- Green
- Red
- RE1
- RE2
- RE3
- NIR
- NIRA
- SWIR1
- SWIR2

### Temporal Dimension

12 monthly observations:

January → December (01–12)

---

## Step 2 — Missing Value Handling

### Problem

The competition represents unavailable observations using the sentinel value:

-9999

These values correspond to months with no valid satellite observation.

### Solution

Converted every occurrence of **-9999** into **NaN** before any calculations.

### Reason

This allows statistical operations such as:

- Mean
- Standard deviation
- Median
- Minimum
- Maximum

to ignore missing months automatically.

---

## Step 3 — Spectral Index Computation

Computed three well-established remote sensing indices for every month.

### NDVI

Measures vegetation density.

Formula:

NDVI = (NIR − Red) / (NIR + Red)

Expected behavior:

- Water → Low or negative
- Vegetation → Positive

---

### NDWI

Measures water content.

Formula:

NDWI = (Green − NIR) / (Green + NIR)

Expected behavior:

- Water → Higher values
- Vegetation → Lower values

---

### MNDWI

Enhanced water index.

Formula:

MNDWI = (Green − SWIR1) / (Green + SWIR1)

Expected behavior:

- Improves water detection.
- More robust near mixed land surfaces.

---

### Results

Generated:

- 12 NDVI features
- 12 NDWI features
- 12 MNDWI features

Total:

36 new spectral features.

Dataset size increased from:

146 → 187 columns.

---

## Step 4 — Temporal Feature Aggregation

### Objective

Reduce dependence on fixed months by summarizing observations across the available seasonal window.

### Statistics Generated

For every raw band and every spectral index:

- Mean
- Standard Deviation
- Minimum
- Maximum
- Median
- Range
- Valid Month Count

### Feature Groups

Aggregated:

- 12 satellite bands
- 3 spectral indices

Total groups:

15

Statistics per group:

7

Generated features:

15 × 7 = 105

---

## Step 5 — Competition-Grade Temporal Features

The aggregation pipeline was further improved by introducing more descriptive statistics.

### Additional Features

#### Coefficient of Variation (CV)

CV = Std / Mean

Purpose:

Measures relative seasonal variability.

Useful because:

- Aquaculture ponds are expected to remain relatively stable.
- Vegetation changes substantially throughout the year.

---

#### Interquartile Range (IQR)

Measures the spread of the middle 50% of observations.

Advantages:

- Less sensitive to extreme values.
- Captures seasonal variability more robustly than the standard deviation.

### New Features Generated

Added:

- CV
- IQR

For every feature group.

Total additional features:

30

---

## Final Feature Space

After feature engineering:

### Original Features

144

### Spectral Indices

36

### Temporal Statistics

135

Total engineered feature space:

315 predictive features

(excluding ID and label).

---

## Train/Test Consistency Verification

### Objective

Ensure identical preprocessing between training and testing datasets.

### Result

Train and Test produced exactly the same engineered feature set.

Verification:

Missing in test : set()

Missing in train: set()

### Conclusion

The feature engineering pipeline is fully reusable and consistent for both datasets.

---

# Feature Selection

## Stage 1 — Constant Feature Removal

### Objective

Remove features with zero variance since they contain no discriminative information.

### Method

Applied:

VarianceThreshold(threshold=0)

### Results

Original Features:

315

Remaining Features:

300

Removed Features:

15

### Removed Features

All removed variables were:

*_valid_months

including:

- VH_valid_months
- VV_valid_months
- Blue_valid_months
- Green_valid_months
- Red_valid_months
- RE1_valid_months
- RE2_valid_months
- RE3_valid_months
- NIR_valid_months
- NIRA_valid_months
- SWIR1_valid_months
- SWIR2_valid_months
- NDVI_valid_months
- NDWI_valid_months
- MNDWI_valid_months

### Interpretation

Every training sample contains observations for all 12 months.

Therefore:

Valid Month Count = 12

for every sample.

These features have zero variance and provide no information to the model.

### Important Note

These features are expected to become informative on the **test set**, where only 4–6 months are available. However, since they are constant in the training data, they cannot contribute to model learning and were removed during feature selection.